In [7]:
import json

# file paths
file1 = "xlam_function_calling_13k_train_processed.json"
file2 = "apigen_mt_5k_train_convert_to_json.json"
output = "apigen_train_merge.json"

# load JSON data
with open(file1, "r", encoding="utf-8") as f1:
    data1 = json.load(f1)

with open(file2, "r", encoding="utf-8") as f2:
    data2 = json.load(f2)

# append contents of 2.json to 1.json
if isinstance(data1, list) and isinstance(data2, list):
    data1.extend(data2)
else:
    raise ValueError("Both JSON files must contain lists to append.")

# write back to 1.json
with open(output, "w", encoding="utf-8") as f1:
    json.dump(data1, f1, ensure_ascii=False, indent=2)

print("Merged 2.json into 1.json successfully.")


Merged 2.json into 1.json successfully.


In [8]:
import transformers
print(transformers.__version__)

/home/yuyao/anaconda3/envs/Qwen_tool/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.51.3


In [5]:
import json
with open("/home/yuyao/LLaMA-Factory/data/train_cleaned.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of items:", len(data))

Number of items: 1589


In [16]:
import json
from collections import Counter

path = "/home/yuyao/LLaMA-Factory/data/dialogue_train2.json"

def load_any(p):
    with open(p, "rb") as f:
        head = f.read(100).lstrip()
    if head.startswith(b"["):
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            return json.load(f)          # JSON array
    else:
        rows = []
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))  # NDJSON
        return rows

rows = load_any(path)

elem_types = Counter()
keysets = Counter()
bad_samples = []  # (row_idx, item_idx, typename, preview)

for i, r in enumerate(rows):
    conv = r.get("conversations", [])
    if not isinstance(conv, list):
        bad_samples.append((i, None, type(conv).__name__, str(conv)[:120]))
        continue
    for j, it in enumerate(conv):
        t = type(it).__name__
        elem_types[t] += 1
        if isinstance(it, dict):
            keysets[tuple(sorted(it.keys()))] += 1
            # content type check
            if "content" in it and not isinstance(it["content"], str):
                bad_samples.append((i, j, f"content:{type(it['content']).__name__}", str(it["content"])[:120]))
        else:
            bad_samples.append((i, j, t, str(it)[:120]))

print("Element types inside conversations:", elem_types)
print("Top keysets:", keysets.most_common(5))
print("First 5 bad samples:", bad_samples[:5])


Element types inside conversations: Counter({'dict': 17600})
Top keysets: [(('from', 'value'), 17600)]
First 5 bad samples: []


In [17]:
import json
from collections import Counter

path = "/home/yuyao/LLaMA-Factory/data/dialogue_train2.json"
rows = json.load(open(path, "r", encoding="utf-8"))

vt = Counter()
bad_samples = []
for i, r in enumerate(rows):
    for j, m in enumerate(r.get("conversations", [])):
        v = m.get("value")
        vt[type(v).__name__] += 1
        if not isinstance(v, str):
            if len(bad_samples) < 5:
                bad_samples.append((i, j, type(v).__name__, str(v)[:120]))

print("value() types:", vt)
print("first bad samples:", bad_samples)


value() types: Counter({'str': 13200, 'list': 4400})
first bad samples: [(0, 2, 'list', "[{'Time': '2024-02-03 09:24:25', 'event_type': 'meal', 'answer': 25.0}, {'Time': '2024-02-03 11:39:25', 'event_type': 'm"), (1, 2, 'list', "[{'Time': '2024-02-03 09:24:25', 'event_type': 'meal', 'answer': 25.0}, {'Time': '2024-02-03 11:39:25', 'event_type': 'm"), (2, 2, 'list', "[{'Time': '2024-02-03 09:24:25', 'event_type': 'meal', 'answer': 25.0}, {'Time': '2024-02-03 11:39:25', 'event_type': 'm"), (3, 2, 'list', "[{'Time': '2024-02-03 09:24:25', 'event_type': 'meal', 'answer': 25.0}, {'Time': '2024-02-03 11:39:25', 'event_type': 'm"), (4, 2, 'list', "[{'Time': '2024-02-23 08:04:25', 'event_type': 'meal', 'answer': 20.0}, {'Time': '2024-02-23 20:09:25', 'event_type': 'm")]


In [1]:
# apigen data preparation
from datasets import load_dataset
ds = load_dataset("Team-ACE/ToolACE")
ds["train"].to_json("./toolACE_train.json", orient="records", lines=True)

/home/yuyao/anaconda3/envs/Qwen_tool/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating json from Arrow format: 100%|██████████| 12/12 [00:00<00:00, 43.99ba/s]


36023467

In [2]:
import json

# Input .jsonl file
input_file = "./toolACE_train.json"
# Output .json file
output_file = "./toolACE_train.json"

data = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  # skip empty lines
            data.append(json.loads(line))

# Save as a JSON array
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Converted {input_file} → {output_file}")


Converted ./toolACE_train.json → ./toolACE_train.json


In [ ]:
from datasets import load_dataset
datasets = load_dataset("Salesforce/APIGen-MT-5k")
datasets["train"].to_json("./apigen_mt_5k_train.json", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  4.67ba/s]


124511899

In [5]:
import json
import random

# Path to your original big JSONL file
input_file = "./xlam_function_calling_60k_train.json"
output_file = "./xlam_function_calling_13k_train.json"

# Read all lines (each line is a JSON object)
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Randomly sample 30,000 lines
sampled_lines = random.sample(lines, 30000)

# Write sampled lines to new file
with open(output_file, "w", encoding="utf-8") as f:
    f.writelines(sampled_lines)

print(f"Saved 13k random samples to {output_file}")


Saved 13k random samples to ./xlam_function_calling_13k_train.json


In [11]:
import json
from typing import List, Dict, Any

INPUT_PATH = "apigen_mt_5k_train.json"          # your 5k file (JSON array or NDJSON)
OUTPUT_PATH = "apigen_mt_5k_train_converted.json"  # will be a JSON array

GENERIC_SYSTEM = (
    "You are a tool-using assistant.\n"
    "- Call at most one tool per turn using a function_call message.\n"
    "- If you call a tool, do not speak to the user in the same turn.\n"
    "- Ask for missing parameters if needed. Keep responses concise.\n"
)

def load_records(path: str) -> List[Dict[str, Any]]:
    """
    Load either a JSON array file or a NDJSON/jsonl file into a list of dicts.
    """
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    # Try JSON array first
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            # Single object file -> wrap into list
            return [data]
    except Exception:
        pass

    # Fallback: NDJSON
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                # skip malformed lines
                continue
    return records

def convert_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    """
    - Change conversations[].from == 'function_call' -> 'gpt'
    - Keep 'observation' as is
    - Replace 'system' with GENERIC_SYSTEM
    - Keep 'tools' as is
    """
    conversations = rec.get("conversations", [])
    new_convs = []
    for msg in conversations:
        msg_from = msg.get("from")
        if msg_from == "function_call":
            # Convert to gpt; keep the original value as-is
            new_convs.append({
                "from": "gpt",
                "value": msg.get("value", "")
            })
        else:
            # keep other messages unchanged (human, gpt, observation, etc.)
            new_convs.append(msg)

    out = dict(rec)  # shallow copy
    out["conversations"] = new_convs
    out["system"] = GENERIC_SYSTEM
    # 'tools' is preserved as-is (if absent, that’s fine)
    return out

def main():
    records = load_records(INPUT_PATH)
    converted = [convert_record(r) for r in records]

    with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
        json.dump(converted, fout, ensure_ascii=False, indent=2)

    print(f"Converted {len(converted)} records -> { OUTPUT_PATH }")

if __name__ == "__main__":
    main()


Converted 5000 records -> apigen_mt_5k_train_converted.json


In [8]:
import json

jsonl_path = "apigen_mt_5k_train.json"
json_path = "apigen_mt_5k_train_convert_to_json.json"

# 读取 jsonl 文件
data = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  # 跳过空行
            data.append(json.loads(line))

# 保存为标准 json 文件
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Converted {len(data)} items from {jsonl_path} to {json_path}")


Converted 5000 items from apigen_mt_5k_train.json to apigen_mt_5k_train_convert_to_json.json


In [6]:
import json
from typing import Any, Dict, List

# ---- paths ----
INPUT_JSONL = "xlam_function_calling_13k_train.json"   # original NDJSON (one object per line)
OUTPUT_JSON = "xlam_function_calling_13k_train_processed.json"  # will be a single JSON array
# ---------------

GENERIC_SYSTEM = (
    "You are a tool-using assistant.\n"
    "- Call at most one tool per turn using a function_call message.\n"
    "- If you call a tool, do not speak to the user in the same turn.\n"
    "- Ask for missing parameters if needed. Keep responses concise.\n"
)

def _parse_json_string(s: str) -> Any:
    """Parse a JSON-encoded string; be forgiving with backslashes."""
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(s.replace("\\\\", "\\"))
        except Exception:
            return None

def _stringify_as_json(val: Any) -> str:
    """
    Return a JSON-encoded string for val.
    - If val is already a JSON string, parse then re-dump (normalized).
    - If val is list/dict, dump.
    - Otherwise -> "[]".
    """
    if val is None:
        return "[]"
    if isinstance(val, str):
        parsed = _parse_json_string(val)
        if parsed is None:
            return "[]"
        return json.dumps(parsed, ensure_ascii=False)
    if isinstance(val, (list, dict)):
        return json.dumps(val, ensure_ascii=False)
    return "[]"

def convert_one(raw: Dict[str, Any]) -> Dict[str, Any]:
    """
    Build:
      conversations = [
        {"from":"human","value": <query>},
        {"from":"gpt","value": <answers_as_json_string>}   # whole array in one message
      ]
      tools = "<JSON string like '[{...}]'>"
      system = GENERIC_SYSTEM
    """
    query = (raw.get("query") or "").strip()
    answers_json_str = _stringify_as_json(raw.get("answers"))
    tools_json_str   = _stringify_as_json(raw.get("tools"))

    conversations = [{"from": "human", "value": query if query else "Hi, I have a request."}]
    # Always put the whole answers array/object in a single gpt message
    conversations.append({"from": "gpt", "value": answers_json_str})

    return {
        "conversations": conversations,
        "tools": tools_json_str,
        "system": GENERIC_SYSTEM,
    }

def main():
    out_items: List[Dict[str, Any]] = []
    total = 0
    bad = 0

    with open(INPUT_JSONL, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue
            out_items.append(convert_one(obj))

    with open(OUTPUT_JSON, "w", encoding="utf-8") as fout:
        json.dump(out_items, fout, ensure_ascii=False, indent=2)

    print(f"Read {total} lines. Wrote {len(out_items)} items to {OUTPUT_JSON}. Skipped malformed: {bad}")

if __name__ == "__main__":
    main()


Read 30000 lines. Wrote 30000 items to xlam_function_calling_13k_train_processed.json. Skipped malformed: 0


In [5]:
import json
import re
from pathlib import Path

# ==== MODIFY THESE ====
input_path  = Path("toolACE_train.json")              # your one raw JSON file (root should be a list of items)
output_path = Path("toolACE_train_sharegpt.json")    # one converted JSON file

# ---- Helpers ----
FUNC_CALL_REGEXES = [
    re.compile(r'^\s*\[.*\([^)]*\).*\]\s*$', re.DOTALL),              # [func(params)] or [func(...), func(...)]
    re.compile(r'^\s*\[\s*\{.*"name"\s*:\s*".*?".*\}\s*\]\s*$', re.DOTALL),  # [{"name": "...", ...}]
]

def looks_like_function_call(txt):
    if not isinstance(txt, str):
        return False
    return any(rx.search(txt) for rx in FUNC_CALL_REGEXES)

def map_role(raw_from, value):
    f = (raw_from or "").lower()
    if f in {"user", "human"}:
        return "human"
    if f in {"tool", "observation"}:
        return "observation"
    if f in {"assistant", "gpt"}:
        return "function_call" if looks_like_function_call(str(value)) else "gpt"
    # fallback
    return "gpt"

def enforce_parity(convos):
    """
    Ensure odd positions (1,3,5,...) are human/observation,
    even positions (2,4,6,...) are gpt/function_call.
    Insert minimal placeholders when needed.
    """
    fixed = []
    pos = 1
    for msg in convos:
        role = msg["from"]
        if role in {"human", "observation"} and pos % 2 == 0:
            fixed.append({"from": "gpt", "value": ""})
            pos += 1
        elif role in {"gpt", "function_call"} and pos % 2 == 1:
            fixed.append({"from": "human", "value": ""})
            pos += 1
        fixed.append(msg)
        pos += 1
    return fixed

def convert_item(item):
    raw_convos = item.get("conversations", [])
    out_convos = []
    for turn in raw_convos:
        new_role = map_role(turn.get("from"), turn.get("value"))
        out_convos.append({"from": new_role, "value": turn.get("value")})
    out_convos = enforce_parity(out_convos)

    out = {"conversations": out_convos}
    if "system" in item:
        out["system"] = item["system"]
    if "tools" in item:
        out["tools"] = item["tools"]
    return out

# ---- Load, convert, save ----
with input_path.open("r", encoding="utf-8") as f:
    root = json.load(f)

# Normalize to list
items = root if isinstance(root, list) else [root]

converted = [convert_item(it) for it in items]

with output_path.open("w", encoding="utf-8") as f:
    json.dump(converted, f, ensure_ascii=False, indent=2)

print(f"Converted {len(items)} items → {output_path}")

Converted 11300 items → toolACE_train_sharegpt.json


In [ ]:
import json
import re
from pathlib import Path

# ==== MODIFY THESE ====
input_path  = Path("toolACE_train_sharegpt.json")              # your one raw JSON file (list of items)
output_path = Path("toolACE_train_sharegpt.json")    # single converted JSON file

# ---- Helpers ----
FUNC_CALL_REGEXES = [
    re.compile(r'^\s*\[.*\([^)]*\).*\]\s*$', re.DOTALL),              # [func(params)]
    re.compile(r'^\s*\[\s*\{.*"name"\s*:\s*".*?".*\}\s*\]\s*$', re.DOTALL),  # [{"name": ...}]
]

def looks_like_function_call(txt):
    if not isinstance(txt, str):
        return False
    return any(rx.search(txt) for rx in FUNC_CALL_REGEXES)

def map_role(raw_from, value):
    f = (raw_from or "").lower()
    if f in {"user", "human"}:
        return "human"
    if f in {"tool", "observation"}:
        return "observation"
    if f in {"assistant", "gpt"}:
        return "function_call" if looks_like_function_call(str(value)) else "gpt"
    return "gpt"

def enforce_parity(convos):
    """Ensure odd turns = human/observation, even turns = gpt/function_call"""
    fixed = []
    pos = 1
    for msg in convos:
        role = msg["from"]
        if role in {"human", "observation"} and pos % 2 == 0:
            fixed.append({"from": "gpt", "value": ""})
            pos += 1
        elif role in {"gpt", "function_call"} and pos % 2 == 1:
            fixed.append({"from": "human", "value": ""})
            pos += 1
        fixed.append(msg)
        pos += 1
    return fixed

def convert_item(item):
    raw_convos = item.get("conversations", [])
    out_convos = []
    for turn in raw_convos:
        new_role = map_role(turn.get("from"), turn.get("value"))
        out_convos.append({"from": new_role, "value": turn.get("value")})
    out_convos = enforce_parity(out_convos)

    out = {"conversations": out_convos}
    if "system" in item:
        out["system"] = item["system"]
    # Always keep tools key at the end
    out["tools"] = item.get("tools", "")
    return out

# ---- Load, convert, save ----
with input_path.open("r", encoding="utf-8") as f:
    root = json.load(f)

items = root if isinstance(root, list) else [root]
converted = [convert_item(it) for it in items]

with output_path.open("w", encoding="utf-8") as f:
    json.dump(converted, f, ensure_ascii=False, indent=2)

print(f"Converted {len(items)} items → {output_path}")


In [9]:
import json
from transformers import AutoTokenizer
import numpy as np

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# path to your json file
json_path = "apigen_train_merge.json"
output_path = "apigen_train_merge_filtered_4000.json"

# load json
with open(json_path, "r") as f:
    data = json.load(f)

lengths = []
filtered_data = []

for item in data:
    # concatenate all conversation values into one string
    text = " ".join(conv["value"] for conv in item.get("conversations", []))
    # tokenize
    tokens = tokenizer(text)["input_ids"]
    length = len(tokens)
    lengths.append(length)

    # keep only items <= 4000
    if length <= 4000:
        filtered_data.append(item)

# compute stats
mean_length = np.mean(lengths)
max_length = np.max(lengths)
median_length = np.median(lengths)
qurtile_90 = np.percentile(lengths, 90)

print(f"Mean token length: {mean_length:.2f}")
print(f"Max token length: {max_length}")
print(f"Median token length: {median_length}")
print(f"90th percentile token length: {qurtile_90}")
print(f"Original items: {len(data)}, Filtered items: {len(filtered_data)}")

# save filtered json
with open(output_path, "w") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)


Mean token length: 431.99
Max token length: 14485
Median token length: 80.0
90th percentile token length: 1867.2000000000044
Original items: 35000, Filtered items: 34638


In [8]:
import json
from transformers import AutoTokenizer
import numpy as np

# load llama-3.1-8B tokenizer
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
# path to your json file
json_path = "apigen_train_merge.json"

# load json
with open(json_path, "r") as f:
    data = json.load(f)

lengths = []

for item in data:
    # concatenate all conversation values into one string
    text = " ".join(conv["value"] for conv in item.get("conversations", []))
    # tokenize
    tokens = tokenizer(text)["input_ids"]
    lengths.append(len(tokens))

# compute mean and max
mean_length = np.mean(lengths)
max_length = np.max(lengths)
median_length = np.median(lengths)
qurtile_90 = np.percentile(lengths, 90)

print(f"Mean token length: {mean_length:.2f}")
print(f"Max token length: {max_length}")
print(f"Median token length: {median_length}")
print(f"90th percentile token length: {qurtile_90}")

/home/yuyao/anaconda3/envs/Qwen_tool/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mean token length: 431.99
Max token length: 14485
Median token length: 80.0
90th percentile token length: 1867.2000000000044


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import json, math
from pathlib import Path

# ======= 配置你的源/目标文件路径 =======
SRC = Path("/home/yuyao/LLaMA-Factory/data/train.json")
DST = Path("/home/yuyao/LLaMA-Factory/data/train_cleaned.json")
BAK = SRC.with_suffix(".bak")

# 保留除 conversations/system/tools 外的其他字段
PRESERVE_EXTRA_FIELDS = True

def is_nan(x):
    return isinstance(x, float) and math.isnan(x)

def to_json_str(obj):
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return str(obj)

def normalize_conversations(value, sample_idx, stats, bad_log):
    """
    目标：返回 list[dict]；失败则返回 None
    支持：list / dict / 字符串（JSON）/ None / NaN
    """
    # 1) 顶层归一为 list
    conv = value
    if isinstance(conv, list):
        pass
    elif isinstance(conv, dict):
        conv = [conv]
        stats["fixed_conv_top"] += 1
    elif isinstance(conv, str):
        try:
            parsed = json.loads(conv)
            if isinstance(parsed, list):
                conv = parsed
                stats["fixed_conv_top"] += 1
            elif isinstance(parsed, dict):
                conv = [parsed]
                stats["fixed_conv_top"] += 1
            else:
                bad_log.append((sample_idx, "conversations string is not list/dict"))
                return None
        except Exception:
            bad_log.append((sample_idx, "conversations invalid json string"))
            return None
    elif conv is None or is_nan(conv):
        bad_log.append((sample_idx, "conversations missing/NaN"))
        return None
    else:
        bad_log.append((sample_idx, f"conversations bad type={type(conv).__name__}"))
        return None

    # 2) 逐 turn 规范
    new_conv = []
    for t in conv:
        if not isinstance(t, dict):
            bad_log.append((sample_idx, "turn not dict"))
            return None

        if "from" not in t or "value" not in t:
            bad_log.append((sample_idx, "turn missing 'from' or 'value'"))
            return None

        role = t["from"]
        val = t["value"]

        # 统一把 value 转成字符串（避免列混类型）
        # 对 function_call 再做额外规范
        if role == "function_call":
            # value 允许是 json 字符串 或 dict
            if isinstance(val, str):
                try:
                    obj = json.loads(val)
                except Exception:
                    # 给一个兜底的空函数
                    obj = {"name": "UnknownFunction", "arguments": {}}
                    stats["fcall_broken_to_stub"] += 1
            elif isinstance(val, dict):
                obj = val
            else:
                # 其他类型 → 兜底
                obj = {"name": "UnknownFunction", "arguments": {}}
                stats["fcall_broken_to_stub"] += 1

            if not isinstance(obj, dict):
                obj = {"name": "UnknownFunction", "arguments": {}}
                stats["fcall_broken_to_stub"] += 1

            # 旧键名兼容：parameters -> arguments
            if "arguments" not in obj and "parameters" in obj:
                obj["arguments"] = obj.pop("parameters")
                stats["fcall_renamed_params"] += 1

            # 自动补齐缺失
            if "name" not in obj or not isinstance(obj["name"], str) or not obj["name"]:
                obj["name"] = "UnknownFunction"
                stats["fcall_filled_name"] += 1

            if "arguments" not in obj or not isinstance(obj["arguments"], dict):
                obj["arguments"] = {}
                stats["fcall_filled_args"] += 1

            new_val = to_json_str(obj)
            stats["normalized_value"] += 1

        else:
            # 非 function_call：
            if isinstance(val, (dict, list)):
                new_val = to_json_str(val)
                stats["normalized_value"] += 1
            elif val is None or is_nan(val):
                new_val = ""
                stats["normalized_value"] += 1
            else:
                # 统一成字符串
                new_val = str(val)
                if not isinstance(val, str):
                    stats["normalized_value"] += 1

        new_conv.append({"from": role, "value": new_val})

    return new_conv

def main():
    if not BAK.exists():
        BAK.write_text(SRC.read_text())

    data = json.loads(SRC.read_text())

    stats = {
        "fixed_conv_top": 0,
        "normalized_value": 0,
        "fcall_renamed_params": 0,
        "fcall_filled_name": 0,
        "fcall_filled_args": 0,
        "fcall_broken_to_stub": 0,
        "fixed_tools_to_str": 0,
        "filled_system_to_str": 0,
        "dropped": 0,
    }
    bad_log = []
    cleaned = []

    for i, item in enumerate(data):
        # conversations 归一
        conv = normalize_conversations(item.get("conversations", None), i, stats, bad_log)
        if conv is None:
            stats["dropped"] += 1
            continue

        out = {}
        # 关键字段：统一类型
        out["conversations"] = conv

        # system：没有就设为字符串空串；不是字符串则转字符串
        sys_val = item.get("system", "")
        if sys_val is None or is_nan(sys_val):
            sys_val = ""
            stats["filled_system_to_str"] += 1
        elif not isinstance(sys_val, str):
            sys_val = to_json_str(sys_val)
            stats["filled_system_to_str"] += 1
        out["system"] = sys_val

        # tools：统一为字符串（你当前就是字符串；某些样本若为对象/数组 → 转为字符串）
        tools_val = item.get("tools", "")
        if tools_val is None or is_nan(tools_val):
            tools_val = ""
            stats["fixed_tools_to_str"] += 1
        elif not isinstance(tools_val, str):
            tools_val = to_json_str(tools_val)
            stats["fixed_tools_to_str"] += 1
        out["tools"] = tools_val

        # 其他字段（可选保留）
        if PRESERVE_EXTRA_FIELDS:
            for k, v in item.items():
                if k in ("conversations", "system", "tools"):
                    continue
                # 尽量不改变其他字段，但注意 NaN → None（更稳）
                if is_nan(v):
                    v = None
                out[k] = v

        cleaned.append(out)

    # 输出文件
    DST.write_text(json.dumps(cleaned, ensure_ascii=False, indent=2))

    # 统计
    print("=== CLEAN SUMMARY ===")
    print(f"Input:  {len(data)}")
    print(f"Output: {len(cleaned)}")
    print(f"Dropped: {stats['dropped']}")
    print(f"fixed_conv_top: {stats['fixed_conv_top']}")
    print(f"normalized_value: {stats['normalized_value']}")
    print(f"function_call: renamed parameters→arguments: {stats['fcALL_renamed_params'] if 'fcALL_renamed_params' in stats else stats['fcall_renamed_params']}")
    print(f"function_call: filled name: {stats['fcALL_filled_name'] if 'fcALL_filled_name' in stats else stats['fcall_filled_name']}")
    print(f"function_call: filled arguments: {stats['fcALL_filled_args'] if 'fcALL_filled_args' in stats else stats['fcall_filled_args']}")
    print(f"function_call: broken→stub: {stats['fcall_broken_to_stub']}")
    print(f"fixed_tools_to_str: {stats['fixed_tools_to_str']}")
    print(f"filled_system_to_str: {stats['filled_system_to_str']}")
    if bad_log:
        print("First 20 bad samples (index, reason):", bad_log[:20])
    print("Saved =>", DST)

if __name__ == "__main__":
    main()


=== CLEAN SUMMARY ===
Input:  1589
Output: 1589
Dropped: 0
fixed_conv_top: 0
normalized_value: 8528
function_call: renamed parameters→arguments: 4
function_call: filled name: 0
function_call: filled arguments: 0
function_call: broken→stub: 0
fixed_tools_to_str: 0
filled_system_to_str: 0
Saved => /home/yuyao/LLaMA-Factory/data/train_cleaned.json


In [3]:
import json, math, numpy as np
from pathlib import Path
SRC = Path("/home/yuyao/LLaMA-Factory/data/train.json")
data = json.loads(SRC.read_text())

bad = []
for i, item in enumerate(data):
    v = item.get("conversations", None)
    t = type(v).__name__
    is_list = isinstance(v, list)
    is_str = isinstance(v, str)
    is_dict = isinstance(v, dict)
    is_none = v is None
    is_nan = isinstance(v, float) and math.isnan(v)
    if not is_list:
        bad.append((i, t, (str(v)[:150] if is_str else t)))

print("bad count =", len(bad))
for i, t, preview in bad[:50]:
    print(f"[{i}] type={t} preview={preview}")


bad count = 0


In [1]:
import json
from pathlib import Path

ODD_ALLOWED = {"human", "observation"}
EVEN_ALLOWED = {"gpt", "function"}   # function_call 归一化成 function

def normalize_role(role: str) -> str:
    r = (role or "").strip().lower()
    if r == "function_call":
        return "function"
    return r

def check_conversations(conv):
    violations = []
    for i, turn in enumerate(conv, start=1):
        role_raw = turn.get("from", "")
        role = normalize_role(role_raw)
        if i % 2 == 1:  # 奇数
            if role not in ODD_ALLOWED:
                violations.append((i, role_raw, f"odd expects {sorted(ODD_ALLOWED)}"))
        else:           # 偶数
            if role not in EVEN_ALLOWED:
                violations.append((i, role_raw, f"even expects {sorted(EVEN_ALLOWED)}"))
    return violations

def load_dataset(path: Path):
    if path.suffix.lower() == ".jsonl":
        items = []
        with path.open("r", encoding="utf-8") as f:
            for ln, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                try:
                    items.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"[WARN] JSONL 第 {ln} 行解析失败: {e}")
        return items
    else:  # .json
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)
        return obj if isinstance(obj, list) else [obj]

# ===== 修改这里的路径 =====
dataset_path = Path("train.json")

data = load_dataset(dataset_path)
total_items = len(data)
total_violations = 0
items_with_violations = 0

for idx, item in enumerate(data, start=1):
    conv = item.get("conversations", [])
    if not isinstance(conv, list):
        print(f"[Item {idx}] conversations 不是列表。")
        items_with_violations += 1
        total_violations += 1
        continue

    violations = check_conversations(conv)
    if violations:
        items_with_violations += 1
        print(f"\n[Item {idx}] 发现 {len(violations)} 处违规：")
        for pos, role, expect in violations:
            print(f"  - 位置 {pos}: from = '{role}', {expect}")
        total_violations += len(violations)

print("\n=== 检查完成 ===")
print(f"总样本数: {total_items}")
print(f"含违规样本数: {items_with_violations}")
print(f"违规总计: {total_violations}")
print(f"合规率: {(total_items - items_with_violations) / total_items * 100:.2f}%")



=== 检查完成 ===
总样本数: 1589
含违规样本数: 0
违规总计: 0
合规率: 100.00%
